# 06 — Final pipeline

Start: the best single model from the comparisons (default: **YOLOv8s + P2 head, input 1280**,
change `START` if another run was better). Three blocks, so the effect of each kind of change is
measured on its own:

| Block | What changes | What stays the same |
|---|---|---|
| A. Inference strategies | NMS IoU, larger test size, TTA, SAHI-style tiles | weights |
| B. Data strategies | (1) grey-out ignored regions in train frames, (2) low-resolution augmentation | model, inference |
| C. Model | bigger model: YOLOv8**m** + P2 (with the best data from B) | data, inference |

The final submission uses the best model (A0 / B / C) with the best inference strategy from A.
`FAST=False` additionally retrains that model on twice as many frames for longer (block D).

Selection metric: **test-weighted mAP** = 0.84 · mAP(val) + 0.16 · mAP(val at 680 px),
because ~16% of test frames are low-resolution (680×382 / 960×540) that never appear in train/val.

Approximate time on a T4 with `FAST=True`: A ≈ 0.5 h, B ≈ 1.5 h, C ≈ 1 h.

## 0. Setup
Works on **Kaggle** (add the competition as input, GPU, Internet on), **Google Colab** (GPU T4,
Kaggle token as Colab secret `KAGGLE_API_TOKEN`, or `kaggle.json` in the root of Google Drive) and **a laptop** (Kaggle token in `<repo>/.kaggle_token`, git-ignored;
data and results go to `~/vdet_data` and `~/vdet_work`; without a GPU use it only with `SMOKE = True`).
On Colab all runs are saved to `MyDrive/vdet`, so after a disconnect just run the notebook again:
finished runs are skipped and interrupted runs continue from the last epoch.

In [ ]:
# ---- Settings ----
SMOKE = False   # True: tiny run on a few images, only to check that the notebook works end to end
FAST = True     # True: every 20th train frame and fewer epochs (~2x faster); False: full protocol
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository
RUN_DATA_STRATEGIES = True    # block B
RUN_MODEL_EXPERIMENT = True   # block C
RUN_LONG_FINAL = not FAST     # block D (off in FAST mode)

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules:  # Colab: keep runs and results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'ultralytics==8.4.207', 'pycocotools', 'kaggle>=2.0'], check=True)

# Shared code (src/vdet.py, src/tvdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

# Same frame subset for every model (see the protocol below). First run: ~5 min to prepare.
TRAIN_STEP = 20 if FAST else 10
S = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, smoke=SMOKE)
print({k: S[k] for k in ["platform", "work", "prep"]})

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

NB = "06_final_pipeline"
GT = f"{S['prep']}/val_gt.json"
VAL = vdet.val_items(S)[:20] if SMOKE else vdet.val_items(S)
VAL_LOW = vdet.lowres_val_items(S, 680)[:len(VAL)]
TEST = vdet.test_items(S)[:30] if SMOKE else vdet.test_items(S)

# Best single model from 04_yolo (change if another run was better)
START = dict(name="yolo_p2", model_src="yolov8s-p2.yaml", load_weights="yolov8s.pt", imgsz=1280, batch=4)
IMGSZ = START["imgsz"]
EPOCHS = 1 if SMOKE else (12 if FAST else 20)
COMMON = dict(optimizer="SGD", lr0=0.01, max_det=500, cache=True, deterministic=True,
              fraction=0.05 if SMOKE else 1.0)
COLS = ["test_weighted", "mAP", "mAP_low680", "AP50", "AP_small", "AP_medium", "AP_large"]
results = {}


def evaluate(name, predict_fn, **extra):
    """Evaluate a prediction function on normal and low-resolution val, log and return a table row."""
    m = vdet.coco_eval(GT, predict_fn(VAL))
    m["mAP_low680"] = vdet.coco_eval(GT, predict_fn(VAL_LOW))["mAP"]
    m["test_weighted"] = round(0.84 * m["mAP"] + 0.16 * m["mAP_low680"], 4)
    vdet.log_result(S, NB, name, m, **extra)
    results[name] = m
    return pd.DataFrame([m], index=[name])[COLS]


def train_yolo(name, S_variant, model_src=START["model_src"], load_weights=START["load_weights"], epochs=EPOCHS):
    """Train one YOLO variant with the protocol of 04_yolo (skips finished runs) and load best.pt."""
    run_dir = vdet.ultra_train(S_variant, name, model_src, imgsz=IMGSZ, batch=START["batch"],
                               load_weights=load_weights, epochs=epochs,
                               close_mosaic=max(1, epochs // 4), **COMMON)
    return YOLO(f"{run_dir}/weights/best.pt")


# best.pt of the START run: from 04_yolo outputs (work dir / Kaggle input), otherwise trained here
start_path = vdet.find_file(S, f"runs/{START['name']}/weights/best.pt")
model = YOLO(start_path) if start_path else train_yolo(START["name"], S)
print("start model:", start_path or START["name"])

## A. Inference strategies (same weights, no retraining)

In [ ]:
INFER = {
    "A0_plain":         lambda m, items: vdet.ultra_predict(m, items, imgsz=IMGSZ, iou=0.6),
    "A1_nms_iou0.5":    lambda m, items: vdet.ultra_predict(m, items, imgsz=IMGSZ, iou=0.5),
    "A1_nms_iou0.7":    lambda m, items: vdet.ultra_predict(m, items, imgsz=IMGSZ, iou=0.7),
    "A2_test_size+320": lambda m, items: vdet.ultra_predict(m, items, imgsz=IMGSZ + 320),
    "A3_tta":           lambda m, items: vdet.ultra_predict(m, items, imgsz=IMGSZ, augment=True),
    "A4_tiles2x2+full": lambda m, items: vdet.ultra_tiled_predict(m, items, imgsz=IMGSZ),
}
for name, fn in INFER.items():
    evaluate(name, lambda items, fn=fn: fn(model, items), block="A")
tableA = vdet.results_table({k: v for k, v in results.items() if k.startswith("A")})[COLS]
best_infer = tableA["test_weighted"].astype(float).idxmax()
print("best inference strategy:", best_infer)
tableA

## B. Data strategies (retrain the same model, plain inference)
* **B1 – masked ignored regions**: ignored regions contain real but unlabelled objects; in plain
  training they teach the model that these objects are background. Here they are painted grey in
  train frames (val is unchanged).
* **B2 – low-resolution augmentation**: 30% of train frames are down-scaled to 680 or 960 px and
  up-scaled back, to imitate the low-resolution test frames.

Compare with `A0_plain` (same model, same inference, original data).

In [ ]:
CANDIDATES = {"A0_plain": dict(model=model, mask=False, low=False, src=START["model_src"], load=START["load_weights"])}
if RUN_DATA_STRATEGIES:
    m_mask = train_yolo("B1_mask_ignore", vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, mask_ignore=True, smoke=SMOKE))
    display(evaluate("B1_mask_ignore", lambda items: vdet.ultra_predict(m_mask, items, imgsz=IMGSZ), block="B"))
    CANDIDATES["B1_mask_ignore"] = dict(model=m_mask, mask=True, low=False, src=START["model_src"], load=START["load_weights"])

    m_low = train_yolo("B2_lowres_aug", vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, lowres_frac=0.3, smoke=SMOKE))
    display(evaluate("B2_lowres_aug", lambda items: vdet.ultra_predict(m_low, items, imgsz=IMGSZ), block="B"))
    CANDIDATES["B2_lowres_aug"] = dict(model=m_low, mask=False, low=True, src=START["model_src"], load=START["load_weights"])

## C. Model change: YOLOv8m + P2
Same data as the best of A0 / B1 / B2, same epochs and inference; only the model is bigger
(≈2.5× more parameters). The difference to that run is the effect of the model alone.

In [ ]:
best_data = max(CANDIDATES, key=lambda n: results[n]["test_weighted"])
D = CANDIDATES[best_data]
print("data of:", best_data, "| mask_ignore =", D["mask"], "| lowres_aug =", D["low"])
S_best = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, mask_ignore=D["mask"],
                    lowres_frac=0.3 if D["low"] else 0.0, smoke=SMOKE)
if RUN_MODEL_EXPERIMENT:
    m_big = train_yolo("C_yolov8m_p2", S_best, model_src="yolov8m-p2.yaml", load_weights="yolov8m.pt")
    display(evaluate("C_yolov8m_p2", lambda items: vdet.ultra_predict(m_big, items, imgsz=IMGSZ), block="C"))
    CANDIDATES["C_yolov8m_p2"] = dict(model=m_big, mask=D["mask"], low=D["low"], src="yolov8m-p2.yaml", load="yolov8m.pt")
vdet.results_table({k: results[k] for k in CANDIDATES})[COLS]

## D. Final model (only with `FAST=False`)
The best candidate is retrained on every 5th train frame (2× more data) for 30 epochs.

In [ ]:
final_name = max(CANDIDATES, key=lambda n: results[n]["test_weighted"])
F = CANDIDATES[final_name]
final_model = F["model"]
if RUN_LONG_FINAL:
    S_long = vdet.setup(train_step=5, val_step=5, max_side=1600, mask_ignore=F["mask"],
                        lowres_frac=0.3 if F["low"] else 0.0, smoke=SMOKE)
    final_model = train_yolo("D_final_long", S_long, model_src=F["src"], load_weights=F["load"],
                             epochs=1 if SMOKE else 30)
    display(evaluate("D_final_long", lambda items: vdet.ultra_predict(final_model, items, imgsz=IMGSZ), block="D"))
    final_name = "D_final_long"
display(evaluate(f"final_{final_name}+{best_infer}", lambda items: INFER[best_infer](final_model, items), block="final"))
print("final model:", final_name, "| inference:", best_infer)

## Final submission

In [ ]:
vdet.write_submission(S, INFER[best_infer](final_model, TEST), "final")

## Summary of all models
All results logged by notebooks 02–06 (this work dir, Kaggle inputs, or the repository's `results/` folder).

In [ ]:
allres = vdet.load_results(S)
cols = [c for c in ["notebook", "run", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "FPS",
                    "mAP_low680", "test_weighted"] if c in allres.columns]
summary = allres[cols].sort_values(["notebook", "mAP"])
summary.to_csv(f"{S['work']}/results/summary_all.csv", index=False)
display(summary)

best_per_nb = allres.loc[allres.groupby("notebook")["mAP"].idxmax()].set_index("run")
ax = best_per_nb[["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]].astype(float).plot.bar(figsize=(11, 4), rot=20)
ax.set_title("Best run of each notebook"); plt.tight_layout()
plt.savefig(f"{S['work']}/figs/summary_best_runs.png", dpi=110); plt.show()